# Data Cleaning

**Authors:** Benedikt Prisett & Stijn Diemel

This notebook cleans the `simpsons_script_lines.csv` data for the following data visualizations.

1. Filter to speaking lines only
2. Drop unused columns
3. Join with episode metadata (season, title, rating)
4. Derive sentence count
5. Export clean CSV


## 1. Setup & Load Raw Data


In [6]:
import pandas as pd
import re

scripts = pd.read_csv("../data/raw_data/simpsons_script_lines.csv", low_memory=False)
episodes = pd.read_csv("../data/raw_data/simpsons_episodes.csv")

print(f"Scripts: {scripts.shape[0]} rows, {scripts.shape[1]} columns")
print(f"Episodes: {episodes.shape[0]} rows, {episodes.shape[1]} columns")

Scripts: 158271 rows, 13 columns
Episodes: 600 rows, 14 columns


In [7]:
print("=== Scripts: null counts ===")
print(scripts.isnull().sum())
print(f"\n=== Scripts: dtypes ===")
print(scripts.dtypes)

=== Scripts: null counts ===
id                        0
episode_id                0
number                    0
raw_text                  0
timestamp_in_ms           0
speaking_line             0
character_id          17521
location_id             407
raw_character_text    17522
raw_location_text       408
spoken_words          26159
normalized_text       26184
word_count            26159
dtype: int64

=== Scripts: dtypes ===
id                      int64
episode_id              int64
number                  int64
raw_text               object
timestamp_in_ms        object
speaking_line          object
character_id           object
location_id           float64
raw_character_text     object
raw_location_text      object
spoken_words           object
normalized_text        object
word_count             object
dtype: object


## 2. Filter to Speaking Lines

Non-speaking rows are stage directions and location descriptions (e.g. `(Street: ext. street - establishing - night)`) or non-verbal actions (e.g. `Bart Simpson: (ANGUISHED SCREAM)`). These have no `spoken_words`, no `character` and are not relevant for the our analysis.

**Dropped:** 26,159 rows where `speaking_line != "true"`

- 26,158 rows with `speaking_line = "false"` (expected)
- 1 row with wrong dialogue text (`"Guess what. I also play Frankenstein!"`) in the `speaking_line` column


In [8]:
scripts = scripts[scripts["speaking_line"] == "true"].copy()
print(f"After filtering speaking lines: {scripts.shape[0]} rows")
print(f"\nRemaining nulls in key columns:")
print(scripts[["raw_character_text", "spoken_words", "word_count"]].isnull().sum())

After filtering speaking lines: 132112 rows

Remaining nulls in key columns:
raw_character_text    2
spoken_words          0
word_count            0
dtype: int64


**Additional rows dropped after filtering:**

- **2 rows with NaN character**
- **15 rows with corrupted `word_count`** (non-numeric values from CSV parsing issues)
- **8 rows with outlier `word_count`** (numeric but absurdly high)


In [9]:
before = len(scripts)
scripts = scripts.dropna(subset=["spoken_words", "raw_character_text"])
print(f"Dropped {before - len(scripts)} rows with NaN in spoken_words or character")

# word_count has ~15 rows with non-numeric values due to CSV column misalignment
scripts["word_count"] = pd.to_numeric(scripts["word_count"], errors="coerce")
bad_wc = scripts["word_count"].isna().sum()
scripts = scripts.dropna(subset=["word_count"])
print(f"Dropped {bad_wc} rows with non-numeric word_count")

# 8 rows have timestamp values leaked into word_count (108K-1.15M)
outliers = (scripts["word_count"] > 200).sum()
scripts = scripts[scripts["word_count"] <= 200]
print(f"Dropped {outliers} rows with outlier word_count (> 200)")

# Drop rows with missing location (very few speaking lines lack location)
loc_na = scripts["location_id"].isna().sum()
scripts = scripts.dropna(subset=["location_id"])
print(f"Dropped {loc_na} rows with missing location_id")

print(f"Remaining: {len(scripts)} rows")
print(
    f"\nTotal dropped from 158,271 raw: {158271 - len(scripts)} rows ({(158271 - len(scripts)) / 158271 * 100:.1f}%)"
)

Dropped 2 rows with NaN in spoken_words or character
Dropped 15 rows with non-numeric word_count
Dropped 8 rows with outlier word_count (> 200)
Dropped 377 rows with missing location_id
Remaining: 131710 rows

Total dropped from 158,271 raw: 26561 rows (16.8%)


## 3. Drop Unused Columns & Rename


In [10]:
scripts = scripts.drop(
    columns=[
        "id",
        "raw_text",
        "timestamp_in_ms",
        "speaking_line",
        "character_id",
        "normalized_text",
    ]
)

scripts = scripts.rename(
    columns={
        "raw_character_text": "character",
        "number": "line_number",
        "raw_location_text": "location",
    }
)

# Convert location_id to int (was float due to NaN rows that are now dropped)
scripts["location_id"] = scripts["location_id"].astype(int)

print(f"Columns: {list(scripts.columns)}")

Columns: ['episode_id', 'line_number', 'location_id', 'character', 'location', 'spoken_words', 'word_count']


## 4. Join with Episode Metadata

The script lines `episode_id` matches the episodes `id` (= `number_in_series`). We join to get `season`, `number_in_season`, `title`, and `imdb_rating`. Inner join keeps only episodes present in both datasets (seasons 1–26, 564 episodes).


In [11]:
ep_metadata = episodes[
    ["id", "season", "number_in_season", "title", "imdb_rating"]
].copy()

scripts = scripts.merge(ep_metadata, left_on="episode_id", right_on="id", how="inner")
scripts = scripts.drop(columns=["id"])

print(f"After join: {len(scripts)} rows")
print(f"Seasons covered: {scripts['season'].min()} to {scripts['season'].max()}")
print(f"Unique episodes: {scripts['episode_id'].nunique()}")

After join: 131710 rows
Seasons covered: 1 to 26
Unique episodes: 564


## 5. Add Sentence Count

Split `spoken_words` on sentence-ending punctuation (`.`, `!`, `?`) to count sentences per line.


In [12]:
def count_sentences(text):
    sentences = re.split(r"[.!?]+", text)
    sentences = [s for s in sentences if s.strip()]
    return max(len(sentences), 1)


scripts["sentence_count"] = scripts["spoken_words"].apply(count_sentences)

print(f"Sentence count stats:")
print(scripts["sentence_count"].describe())

Sentence count stats:
count    131710.000000
mean          1.701799
std           1.066504
min           1.000000
25%           1.000000
50%           1.000000
75%           2.000000
max          34.000000
Name: sentence_count, dtype: float64


## 6. Convert Dtypes, Sort & Reorder


In [13]:
scripts["word_count"] = scripts["word_count"].astype(int)
scripts["sentence_count"] = scripts["sentence_count"].astype(int)
scripts["season"] = scripts["season"].astype(int)
scripts["number_in_season"] = scripts["number_in_season"].astype(int)
scripts["line_number"] = scripts["line_number"].astype(int)

scripts = scripts.sort_values(["episode_id", "line_number"]).reset_index(drop=True)

scripts = scripts[
    [
        "episode_id",
        "season",
        "number_in_season",
        "title",
        "imdb_rating",
        "line_number",
        "character",
        "location_id",
        "location",
        "spoken_words",
        "word_count",
        "sentence_count",
    ]
]

print(f"Final columns: {list(scripts.columns)}")

Final columns: ['episode_id', 'season', 'number_in_season', 'title', 'imdb_rating', 'line_number', 'character', 'location_id', 'location', 'spoken_words', 'word_count', 'sentence_count']


## 7. Final Verification


In [14]:
print(f"Shape: {scripts.shape[0]} rows, {scripts.shape[1]} columns")
print(f"Seasons: {scripts['season'].min()} to {scripts['season'].max()}")
print(f"Episodes: {scripts['episode_id'].nunique()}")
print(f"\nNull counts:")
print(scripts.isnull().sum())
print(f"\nDtypes:")
print(scripts.dtypes)
scripts.head(10)

Shape: 131710 rows, 12 columns
Seasons: 1 to 26
Episodes: 564

Null counts:
episode_id          0
season              0
number_in_season    0
title               0
imdb_rating         0
line_number         0
character           0
location_id         0
location            0
spoken_words        0
word_count          0
sentence_count      0
dtype: int64

Dtypes:
episode_id            int64
season                int64
number_in_season      int64
title                object
imdb_rating         float64
line_number           int64
character            object
location_id           int64
location             object
spoken_words         object
word_count            int64
sentence_count        int64
dtype: object


,episode_id,season,number_in_season,title,imdb_rating,line_number,character,location_id,location,spoken_words,word_count,sentence_count
0,1,1,1,Simpsons Roasting on an Open Fire,8.2,2,Marge Simpson,2,Car,"Ooo, careful, Homer.",3,1
1,1,1,1,Simpsons Roasting on an Open Fire,8.2,3,Homer Simpson,2,Car,There's no time to be careful.,6,1
2,1,1,1,Simpsons Roasting on an Open Fire,8.2,4,Homer Simpson,2,Car,We're late.,2,1
3,1,1,1,Simpsons Roasting on an Open Fire,8.2,7,Marge Simpson,4,Auditorium,"Sorry, Excuse us. Pardon me...",5,2
4,1,1,1,Simpsons Roasting on an Open Fire,8.2,8,Homer Simpson,4,Auditorium,"Hey, Norman. How's it going? So you got dragge...",21,6
5,1,1,1,Simpsons Roasting on an Open Fire,8.2,9,Homer Simpson,4,Auditorium,Pardon my galoshes.,3,1
6,1,1,1,Simpsons Roasting on an Open Fire,8.2,10,Seymour Skinner,4,Auditorium,"Wasn't that wonderful? And now, ""Santas of Man...",17,2
7,1,1,1,Simpsons Roasting on an Open Fire,8.2,11,Marge Simpson,4,Auditorium,Oh... Lisa's class.,3,2
8,1,1,1,Simpsons Roasting on an Open Fire,8.2,12,JANEY,4,Auditorium,Frohlich weihnachten -- that's German for Merr...,27,2
9,1,1,1,Simpsons Roasting on an Open Fire,8.2,13,Todd Flanders,4,Auditorium,"Meri Kurimasu. I am Hotseiosha, a Japanese pri...",29,3


## 8. Save Clean Dataset


In [15]:
scripts.to_csv("../data/clean_data/simpsons_script_lines_clean.csv", index=False)
print("Saved to data/clean_data/simpsons_script_lines_clean.csv")

Saved to data/clean_data/simpsons_script_lines_clean.csv
